# Lesson 7 : Workflows

Up until now, we have built a single agent with language models, tools, and messages.<br>
This type of agent essentially performs to call LLMs or tools until it reaches to the goal - such as, [ReAct](https://tsmatz.wordpress.com/2023/03/07/react-with-openai-gpt-and-langchain/)-style pattern. (See the left side in below picture.) But various processing patterns (such as, workflow pattern, fan-out/fan-in pattern, etc) or the mixture of these patterns will be required in production.

![Flow patterns](./assets/flow_patterns.png)

> Note : Especially in complex tasks, cramming all the process into a single agent will result into poor performance.  
> By dividing the roles among multiple agents and collaborating these blocks, the context becomes clear, and accuracy is then expected to improve. (Also, manageability will be improved.)

Using built-in process builders in Agent Framework, you can build various processing patterns to orchestrate your existing agents and custom code.  
```WorkflowBuilder``` is a generic builder for a flexible workflow graph, and other builders (```SequentialBuilder```, ```ConcurrentBuilder```, ```GroupChatBuilder```, ```HandoffBuilder```, and ```MagenticBuilder``` - which are all built with ```WorkflowBuilder```) handles a variety of processing design patterns depending on your needs.

In this exercise, we explore fundamental workflow's capabilities in Agent Framework using the following brief sequential flow.

1. Create a plan for the trip.
2. Revise the generated plan.

> Note : Learning a variety of multi-agent design patterns (orchestration patterns) is out of scope in this repository. (See [official document](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns) for this topic.)

## 1. Build and run a simple workflow

In this example, we build a workflow with generic ```WorkflowBuilder``` class in order to understand how it works internally.

> Note : You can easily build this type of simple sequential flows with built-in ```SequentialBuilder``` class (instead of generic ```WorkflowBuilder```), but we don't use this high-level class for your learning purpose.

First we initialize the client as usual.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Now let's build a workflow and run as follows.

In workflow on Agent Framework, first we register ```Executor``` as building blocks.  
In this example, we use the following 3 ```Executor``` - two is ```AgentExecutor``` (which is generated and registered by ```register_agent()```) and one is generic ```Executor``` (which is registered by ```register_executor()```).

- PlanningAgent : This is an executor by LLM agent to generate a plan.
- ReviseAgent : This is an executor by LLM agent to revise a plan.
- FinalResponseGenerator : This executor generates the final response and finalize the workflow. (The workflow's instance becomes idle state after this executor is run.) In this example, it returns a list of all messages for the final response.

As you can see in below code, a generic ```Executor``` is processed in a class method (function) decorated by ```@handler```. (In a single ```Executor```, you can also define multiple handlers depending on arguments.)  
All executors are connected by workflow context (the argument ```ctx``` in this code), and you can pass result (output) to another executor by using context. (See code in the following ```generate_final_respose()``` handler.)

> Note : It is not recommended to split a simple task into multiple agents, because it might degrade performance.  
> Please avoid overusing multi-agents. (This is for demo purpose.)

In [2]:
from agent_framework import ChatAgent, ChatMessage
from agent_framework import (
    WorkflowBuilder,
    Executor,
    AgentExecutorResponse,
    WorkflowContext,
    handler
)

class ResponseGenerator(Executor):
    @handler
    async def generate_final_respose(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext[list[ChatMessage]]
    ) -> None:
        # in this example, we send full conversation as a final result.
        await ctx.yield_output(list(response.full_conversation))

workflow = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="Your task is to help users plan and concisely summarize it in five bullet points.",  # "ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。"
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="Your task is to review the plan you receive and to refine it further.",  # "与えられた計画を確認して、より洗練させます。"
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: ResponseGenerator(id="final_response_generator"),
        name="FinalResponseGenerator"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalResponseGenerator")
    .set_start_executor("PlanningAgent")
    .build()
)

events = await workflow.run("Create a plan for my travel in Osaka.")  # "大阪の旅行計画を作成してください。"

Now we output the returned result (all messages in this example) as follows.

In [3]:
from agent_framework import Role

outputs = events.get_outputs()
messages = outputs[1]
for i, (msg) in enumerate(messages, start=1):
    author_name = msg.author_name
    if author_name is None:
        author_name = "assistant" if msg.role == Role.ASSISTANT else "user"
    print("----------------------------------")
    print(f"[{i:02d}: {author_name}]")
    print(msg.text)

----------------------------------
[01: user]
Create a plan for my travel in Osaka.
----------------------------------
[02: PlanningAgent]
- **Set your trip basics:** Choose travel dates/length, arrival airport (KIX/ITM), neighborhood to stay (Namba/Umeda/Shin-Osaka), and daily pace (relaxed vs packed).  
- **Core Osaka sights (2–3 days):** Dotonbori & Shinsaibashi, Osaka Castle area, Umeda (Sky Building/Hep Five), Kuromon Market, and a night view/food crawl.  
- **Food-focused plan:** Must-tries (takoyaki, okonomiyaki, kushikatsu, ramen), pick 1–2 “food hubs” per day (Namba, Shinsekai, Tenma), and reserve any high-demand spots.  
- **Day trips to add (1–3 days):** Kyoto (temples + Gion), Nara (deer + Todaiji), Kobe (harbor + beef), or Himeji (castle) based on your interests.  
- **Logistics & bookings:** Get an ICOCA card, map routes on Google Maps, consider passes only if they save money, prebook Universal Studios Japan/teamsLabs? (if relevant), and keep a weather backup plan (aquari

## 2. Run workflow with streaming

You can also run workflow with streaming.  
Especially when you handle checkpointing (pause/resume), human-in-the-loop (HITL), etc, this might be useful. (These topics will be discussed later.)

In [4]:
from agent_framework import WorkflowOutputEvent

async for event in workflow.run_stream("Create a plan for my travel in Osaka."):  # "大阪の旅行計画を作成してください。"
    print(f"[{event}]")
    # if isinstance(event, WorkflowOutputEvent):
    #     print("----------------------------------")
    #     print(event.data)

[WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)]
[WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)]
[ExecutorInvokedEvent(executor_id=PlanningAgent, data=Create a plan for my travel in Osaka.)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=-)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages= **)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=Trip)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages= setup)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages= ()]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=today)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=):)]
[AgentRunUpdateEvent(executor_id=PlanningAge

## 3. Checkpoint

When you run a long-running workflow, it might need to suspend the running and restart it later.  
With checkpoint in workflow on Agent Framework, you can save (serialize) the state in the storage, and restore (deserialize) the state to resume the suspended instance.

In this example, we explore checkpoint using ```InMemoryCheckpointStorage``` (non-persistent storage) for demo purpose.

Checkpoint is created after each super step iteration internally.  
In this example, therefore, we suspend workflow when the first ```SuperStepCompletedEvent``` is captured. And we then restore the checkpoint and resume the workflow.

First we generate a workflow again, with checkpoint enabled.

> Note : When you need to restore custom data in your ```Executor```, please implement ```on_checkpoint_save()``` and ```on_checkpoint_restore()``` methods (override methods) in your custom executor class. (This example doesn't have such data.)

In [5]:
from agent_framework import InMemoryCheckpointStorage

checkpoint_storage = InMemoryCheckpointStorage()

workflow_builder = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="Your task is to help users plan and concisely summarize it in five bullet points.",  # "ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。"
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="Your task is to review the plan you receive and to refine it further.",  # "与えられた計画を確認して、より洗練させます。"
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: ResponseGenerator(id="final_response_generator"),
        name="FinalResponseGenerator"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalResponseGenerator")
    .set_start_executor("PlanningAgent")
    .with_checkpointing(checkpoint_storage=checkpoint_storage)
)

Now let's run and suspend workflow as follows.  
As I have mentioned above, we suspend workflow when the first ```SuperStepCompletedEvent``` is captured.

In [6]:
from agent_framework import SuperStepCompletedEvent

workflow = workflow_builder.build()

async for event in workflow.run_stream("Create a plan for my travel in Osaka."):  # "大阪の旅行計画を作成してください。"
    # we suspend the workflow when the first superstep completion is captured
    if isinstance(event, SuperStepCompletedEvent):
        break

Let's get all checkpoints and show the number of all saved checkpoints.  
Here we have 2 checkpoints - the first checkpoint is initial checkpoint (which is saved if there are messages from initial execution) and the second is created after the first super step iteration.

In [7]:
checkpoints = await checkpoint_storage.list_checkpoints()
print(len(checkpoints))

2


Now let's restore state by using checkpoint and resume workflow as follows.

In [8]:
workflow = workflow_builder.build()

final_checkpoint = checkpoints[-1]
events = await workflow.run(checkpoint_id=final_checkpoint.checkpoint_id)

Let's see the completed result (output) of this workflow.  
Even after suspended and resumed, the output is correctly generated.

In [9]:
outputs = events.get_outputs()
messages = outputs[0]
for i, (msg) in enumerate(messages, start=1):
    author_name = msg.author_name
    if author_name is None:
        author_name = "assistant" if msg.role == Role.ASSISTANT else "user"
    print("----------------------------------")
    print(f"[{i:02d}: {author_name}]")
    print(msg.text)

----------------------------------
[01: user]
Create a plan for my travel in Osaka.
----------------------------------
[02: PlanningAgent]
- **Set basics (dates, budget, base):** Pick travel dates/length, daily budget, and choose where to stay (best bases: **Namba/Shinsaibashi** for food/nightlife, **Umeda** for transport/hubs, **Tennoji** for value).  
- **Core areas by day:** Allocate days to **(1) Namba–Dotonbori–Shinsaibashi**, **(2) Osaka Castle + museum/parks**, **(3) Umeda (Sky Building) + shopping/food halls**, **(4) Bay Area: Kaiyukan Aquarium + Tempozan**, **(5) Shinsekai/Tennoji + Abeno Harukas**.  
- **Day trips (choose 1–2):** Add **Kyoto** (temples), **Nara** (deer + Todaiji), **Kobe** (harbor + beef), or **Himeji** (castle) based on interests and time.  
- **Food & reservations:** Plan key eats (takoyaki, okonomiyaki, kushikatsu, ramen, izakaya), make any must-visit reservations, and set a “street-food evening” around **Dotonbori**.  
- **Logistics checklist:** Get an **